# RAG（検索拡張生成）

大きなドキュメントを扱うために、文書をチャンクに分割し、質問に関連する部分だけをClaudeに渡す技術です。

**実装の流れ：**
1. テキストチャンキング戦略
2. テキスト埋め込み（Embedding）
3. RAGフロー全体の実装

## 1. セットアップ

In [93]:
from dotenv import load_dotenv
load_dotenv()

import re
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

print("セットアップ完了")

セットアップ完了


## 2. テキストチャンキング戦略

ドキュメントをどう分割するかがRAGの品質に直結します。

| 戦略 | 特徴 | 向いているケース |
|---|---|---|
| サイズベース | 固定文字数で分割・重複あり | コード含む任意のコンテンツ |
| 構造ベース | 見出し・セクションで分割 | Markdownなど書式が保証された文書 |
| 文ベース | 文単位で分割・重複あり | 一般的なテキスト文書 |
| 意味ベース | NLPで関連文をグループ化 | 高精度が必要な場合（計算コスト高）|

In [89]:
# サイズベースのチャンキング（重複あり）
# 最もシンプルで信頼性が高く、本番でもよく使われる
def chunk_by_char(text: str, chunk_size: int = 150, chunk_overlap: int = 20) -> list[str]:
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        # 次のチャンクの開始位置：重複分だけ戻る
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)

    return chunks


# 構造ベースのチャンキング（Markdownの ## ヘッダーで分割）
def chunk_by_section(document_text: str) -> list[str]:
    pattern = r"\n## "
    return re.split(pattern, document_text)


# 文ベースのチャンキング（句読点で分割・重複あり）
def chunk_by_sentence(
    text: str, max_sentences_per_chunk: int = 5, overlap_sentences: int = 1
) -> list[str]:
    sentences = re.split(r"(?<=[。.!?])\s*", text)
    sentences = [s for s in sentences if s.strip()]  # 空文字を除外

    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        chunks.append(" ".join(sentences[start_idx:end_idx]))
        start_idx += max_sentences_per_chunk - overlap_sentences
        if start_idx < 0:
            start_idx = 0

    return chunks


print("チャンキング関数の定義完了")

チャンキング関数の定義完了


In [90]:
# サンプルテキストで3つの戦略を比較する
sample_text = """## リスク要因
この会社は市場変動リスクにさらされています。金利の上昇は収益に悪影響を与える可能性があります。また、為替リスクも重要な課題です。

## ソフトウェア開発
エンジニアチームは今年、合計342件のバグを修正しました。コードレビューの強化により、品質が大幅に向上しています。新機能の開発も順調に進んでいます。

## 医学研究部門
研究チームはウイルスのバグ（変異）を追跡しています。新しいワクチンの開発が進んでいます。臨床試験の結果は良好です。"""

print("=== サイズベース（chunk_size=100, overlap=20） ===")
char_chunks = chunk_by_char(sample_text, chunk_size=100, chunk_overlap=20)
for i, c in enumerate(char_chunks):
    print(f"チャンク{i+1}（{len(c)}文字）: {c[:60]}...")

print("\n=== 構造ベース（## で分割） ===")
section_chunks = chunk_by_section(sample_text)
for i, c in enumerate(section_chunks):
    print(f"チャンク{i+1}: {c[:60].strip()}...")

print("\n=== 文ベース（3文ごと・1文重複） ===")
sentence_chunks = chunk_by_sentence(sample_text, max_sentences_per_chunk=3, overlap_sentences=1)
for i, c in enumerate(sentence_chunks):
    print(f"チャンク{i+1}: {c[:80]}...")

=== サイズベース（chunk_size=100, overlap=20） ===
チャンク1（100文字）: ## リスク要因
この会社は市場変動リスクにさらされています。金利の上昇は収益に悪影響を与える可能性があります。また、為...
チャンク2（100文字）: トウェア開発
エンジニアチームは今年、合計342件のバグを修正しました。コードレビューの強化により、品質が大幅に向上して...
チャンク3（70文字）: 。

## 医学研究部門
研究チームはウイルスのバグ（変異）を追跡しています。新しいワクチンの開発が進んでいます。臨床試...

=== 構造ベース（## で分割） ===
チャンク1: ## リスク要因
この会社は市場変動リスクにさらされています。金利の上昇は収益に悪影響を与える可能性があります。また、為...
チャンク2: ソフトウェア開発
エンジニアチームは今年、合計342件のバグを修正しました。コードレビューの強化により、品質が大幅に向上...
チャンク3: 医学研究部門
研究チームはウイルスのバグ（変異）を追跡しています。新しいワクチンの開発が進んでいます。臨床試験の結果は良...

=== 文ベース（3文ごと・1文重複） ===
チャンク1: ## リスク要因
この会社は市場変動リスクにさらされています。 金利の上昇は収益に悪影響を与える可能性があります。 また、為替リスクも重要な課題です。...
チャンク2: また、為替リスクも重要な課題です。 ## ソフトウェア開発
エンジニアチームは今年、合計342件のバグを修正しました。 コードレビューの強化により、品質が大幅に...
チャンク3: コードレビューの強化により、品質が大幅に向上しています。 新機能の開発も順調に進んでいます。 ## 医学研究部門
研究チームはウイルスのバグ（変異）を追跡してい...
チャンク4: ## 医学研究部門
研究チームはウイルスのバグ（変異）を追跡しています。 新しいワクチンの開発が進んでいます。 臨床試験の結果は良好です。...
チャンク5: 臨床試験の結果は良好です。...


## 3. テキスト埋め込み（Embedding）\n\nテキストを数値ベクトルに変換します。意味が近いテキストほど、ベクトル空間での距離が近くなります。\n\nAnthropicは埋め込みAPIを提供していないため、推奨プロバイダーの **VoyageAI** を使用します。\n\n**事前準備：**\n1. https://www.voyageai.com/ でAPIキーを取得\n2. `.env` に `VOYAGE_API_KEY=\"your_key_here\"` を追加\n3. 下のセルで `voyageai` をインストール"

In [91]:
%pip install voyageai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 14.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 18.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 15.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 14.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 17.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.4/640.4 kB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 17.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36/36 [voyageai]/36 [tokenizers]-hub]
Note: you may need to restart the kernel to use updated packages.


In [94]:
import voyageai

voyage_client = voyageai.Client()  # VOYAGE_API_KEY を環境変数から自動読み込み


def generate_embedding(text: str, model: str = "voyage-3-large", input_type: str = "query") -> list[float]:
    """テキストを埋め込みベクトルに変換する
    
    input_type:
      "query"    → 検索クエリ（ユーザーの質問）に使用
      "document" → 保存するチャンクに使用
    """
    result = voyage_client.embed([text], model=model, input_type=input_type)
    return result.embeddings[0]


# 動作確認：埋め込みベクトルの形状を確認する
sample_embedding = generate_embedding("テスト")
print(f"ベクトルの次元数: {len(sample_embedding)}")
print(f"最初の5要素: {sample_embedding[:5]}")
print(f"値の範囲: {min(sample_embedding):.4f} 〜 {max(sample_embedding):.4f}")

ベクトルの次元数: 1024
最初の5要素: [-0.03609152510762215, 0.019363710656762123, -0.015004340559244156, 0.022912034764885902, -0.032847341150045395]
値の範囲: -0.1160 〜 0.0977


## 4. RAGフロー全体

**6ステップの流れ：**
1. ソーステキストをチャンクに分割
2. 各チャンクをEmbeddingに変換
3. ベクターDBに保存（前処理はここまで）
4. ユーザーのクエリをEmbeddingに変換
5. コサイン類似度で最も近いチャンクを検索
6. 関連チャンク＋質問をClaudeに渡して回答生成

**コサイン類似度：**
- 2つのベクトルの「向きの近さ」を -1〜1 で表す
- 1 に近い → 意味が近い
- 0 → 無関係
- -1 → 正反対の意味

In [95]:
import math

# ステップ1：ソーステキスト（チャンク済み）
chunks = [
    "第1章：医学研究 — 今年は、XDR-47ウイルスに関する理解において大きな進歩が見られた。",
    "第2章：ソフトウェアエンジニアリング — 当部門は分散システムにおける様々な感染経路の研究に多大な努力を注いできた。",
]

# ステップ2＆3：各チャンクをEmbeddingに変換してインメモリDBに保存
print("チャンクをEmbeddingに変換中...")
vector_db = []
for chunk in chunks:
    embedding = generate_embedding(chunk, input_type="document")
    vector_db.append({"text": chunk, "embedding": embedding})
    print(f"  保存: {chunk[:40]}... （{len(embedding)}次元）")

print(f"\n前処理完了：{len(vector_db)}件のチャンクを保存")

チャンクをEmbeddingに変換中...
  保存: 第1章：医学研究 — 今年は、XDR-47ウイルスに関する理解において大きな進歩... （1024次元）
  保存: 第2章：ソフトウェアエンジニアリング — 当部門は分散システムにおける様々な感染... （1024次元）

前処理完了：2件のチャンクを保存


In [100]:
def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """2つのベクトルのコサイン類似度を計算する（-1〜1）"""
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    magnitude_a = math.sqrt(sum(a ** 2 for a in vec_a))
    magnitude_b = math.sqrt(sum(b ** 2 for b in vec_b))
    return dot_product / (magnitude_a * magnitude_b)


def search(query: str, vector_db: list, top_k: int = 1) -> list[dict]:
    """クエリに最も近いチャンクをコサイン類似度で検索する"""
    # ステップ4：クエリをEmbeddingに変換（1回だけ）
    query_embedding = generate_embedding(query, input_type="query")

    # ステップ5：全チャンクとコサイン類似度を計算してソート
    results = []
    for item in vector_db:
        similarity = cosine_similarity(query_embedding, item["embedding"])
        results.append({"text": item["text"], "similarity": similarity})

    results = sorted(results, key=lambda x: x["similarity"], reverse=True)

    # 全スコアを表示してから top_k を返す
    print("=== 類似度スコア（全チャンク） ===")
    for item in results:
        print(f"  {item['similarity']:.4f}  {item['text'][:50]}...")

    return results[:top_k]


# 検索テスト（Embeddingは1回だけ生成）
query = "ソフトウェアエンジニアリング部門は今年どのような活動をしましたか？"
print(f"クエリ: {query}\n")
search(query, vector_db)

クエリ: ソフトウェアエンジニアリング部門は今年どのような活動をしましたか？

=== 類似度スコア（全チャンク） ===
  0.4956  第2章：ソフトウェアエンジニアリング — 当部門は分散システムにおける様々な感染経路の研究に多大な努...
  0.2884  第1章：医学研究 — 今年は、XDR-47ウイルスに関する理解において大きな進歩が見られた。...


[{'text': '第2章：ソフトウェアエンジニアリング — 当部門は分散システムにおける様々な感染経路の研究に多大な努力を注いできた。',
  'similarity': 0.49559811097497836}]

In [102]:
def rag(query: str, vector_db: list) -> str:
    """ステップ6：関連チャンク＋質問をClaudeに渡して回答を生成する"""
    # 最も関連性の高いチャンクを1件取得
    top_result = search(query, vector_db, top_k=1)[0]
    print(f"使用チャンク（類似度: {top_result['similarity']:.4f}）:\n  {top_result['text']}\n")

    prompt = f"""以下のドキュメントの内容を元に、ユーザーの質問に回答してください。

<user_question>
{query}
</user_question>

<document>
{top_result['text']}
</document>"""

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


# RAGパイプライン全体を実行
answer = rag(query, vector_db)
print(f"=== Claudeの回答 ===\n{answer}")

RateLimitError: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.

## 5. RAGフロー実装

`VectorIndex` クラスにチャンクとEmbeddingをまとめて管理させます。
`generate_embedding` を文字列リストにも対応させてバッチ処理を効率化します。

In [114]:
def generate_embedding(text, model="voyage-3-large", input_type="query"):
    """テキストを埋め込みベクトルに変換する。文字列・リスト両対応"""
    texts = text if isinstance(text, list) else [text]
    result = voyage_client.embed(texts, model=model, input_type=input_type)
    return result.embeddings if isinstance(text, list) else result.embeddings[0]


class VectorIndex:
    """シンプルなインメモリベクターDB"""

    def __init__(self):
        self.vectors = []
        self.documents = []  # BM25Index と同名にして Retriever から統一アクセス可能に

    def add_document(self, metadata: dict):
        """文書を追加してEmbeddingを生成・保存する"""
        embedding = generate_embedding(metadata["content"], input_type="document")
        self.vectors.append(embedding)
        self.documents.append(metadata)

    def search(self, query: str, top_k: int = 1) -> list[tuple]:
        """クエリテキストをEmbedding変換してコサイン距離で検索する"""
        query_vector = generate_embedding(query, input_type="query")
        results = []
        for vector, meta in zip(self.vectors, self.documents):
            similarity = cosine_similarity(query_vector, vector)
            distance = 1 - similarity
            results.append((meta, distance))
        return sorted(results, key=lambda x: x[1])[:top_k]


print("VectorIndex クラスの定義完了")

VectorIndex クラスの定義完了


In [104]:
# ステップ1：report.md を読み込んでセクションごとに分割
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)
print(f"チャンク数: {len(chunks)}")
print(f"\nチャンク3の内容:\n{chunks[2]}")

チャンク数: 7

チャンク3の内容:
第1章：医学研究
今年は、XDR-47ウイルスに関する理解において大きな進歩が見られた。
新しいワクチン候補が3件特定され、うち1件は臨床試験フェーズ2に進んだ。
研究チームは感染経路の解明に注力し、従来比40%の感染リスク低減を達成した。



In [116]:
# 全チャンクのEmbeddingをバッチ生成してVectorIndexに保存
# バッチでEmbeddingを生成してからまとめて add するため API 呼び出しは1回
embeddings = generate_embedding(chunks, input_type="document")
print(f"生成したEmbedding数: {len(embeddings)}, 各次元数: {len(embeddings[0])}")

store = VectorIndex()
for embedding, chunk in zip(embeddings, chunks):
    # Embeddingは生成済みなので直接セットする（API呼び出しなし）
    store.vectors.append(embedding)
    store.documents.append({"content": chunk})

print(f"ベクターDBに保存完了: {len(store.vectors)}件")

生成したEmbedding数: 7, 各次元数: 1024
ベクターDBに保存完了: 7件


In [106]:
# ステップ4＆5：クエリのEmbeddingを生成して検索
query = "ソフトウェアエンジニアリング部門は昨年どのような活動をしましたか？"
user_embedding = generate_embedding(query, input_type="query")

results = store.search(user_embedding, top_k=2)

print(f"クエリ: {query}\n")
print("=== 検索結果（コサイン距離：0に近いほど類似） ===")
for doc, distance in results:
    print(f"\n距離: {distance:.4f}")
    print(doc["content"][:200])

クエリ: ソフトウェアエンジニアリング部門は昨年どのような活動をしましたか？

=== 検索結果（コサイン距離：0に近いほど類似） ===

距離: 0.4551
第2章：ソフトウェアエンジニアリング
当部門は分散システムにおける安定性向上に多大な努力を注いだ。
エンジニアチームは合計342件のバグを修正し、システム稼働率は99.9%を達成した。
新機能として、リアルタイム監視ダッシュボードとAI駆動の異常検知システムを導入した。
コードレビュープロセスの強化により、本番環境へのデプロイ品質が大幅に向上した。


距離: 0.7092
概要
本報告書は、各部門の2024年度の活動実績をまとめたものです。



## 6. BM25語彙検索

セマンティック検索は「意味の近さ」で検索しますが、完全一致のキーワード（インシデントIDなど）には弱い場合があります。

**BM25（Best Match 25）** は語彙検索アルゴリズムで、以下の特徴があります：
- 稀な特定の用語（例：`INC-2023-Q4-011`）に高い重みを付ける
- 「a」「the」などよく使われる単語は無視
- セマンティック検索と組み合わせて「ハイブリッド検索」を実現

| | セマンティック検索 | BM25語彙検索 |
|---|---|---|
| 得意 | 意味・文脈の類似 | 完全一致・専門用語・ID |
| 不得意 | 完全一致の検索 | 言い換え・同義語 |

In [107]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [110]:
from rank_bm25 import BM25Okapi


def tokenize_ja(text: str) -> list[str]:
    """日本語テキストを文字n-gram（2文字）でトークン化する"""
    text = text.lower()
    # 2文字のn-gramに分割（日本語はスペース区切りがないため）
    tokens = [text[i:i+2] for i in range(len(text) - 1)]
    return tokens if tokens else [text]


class BM25Index:
    """BM25による語彙検索インデックス（日本語対応）"""

    def __init__(self):
        self.documents = []
        self.bm25 = None

    def add_document(self, metadata: dict):
        self.documents.append(metadata)

    def _build(self):
        tokenized = [tokenize_ja(doc["content"]) for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized)

    def search(self, query: str, top_k: int = 1) -> list[tuple]:
        if self.bm25 is None:
            self._build()
        tokenized_query = tokenize_ja(query)
        scores = self.bm25.get_scores(tokenized_query)
        max_score = max(scores) if max(scores) > 0 else 1
        results = [
            (self.documents[i], 1 - scores[i] / max_score)
            for i in range(len(scores))
        ]
        return sorted(results, key=lambda x: x[1])[:top_k]


print("BM25Index クラスの定義完了（日本語対応）")

BM25Index クラスの定義完了（日本語対応）


In [117]:
# BM25インデックスにチャンクを登録
bm25_store = BM25Index()
for chunk in chunks:
    bm25_store.add_document({"content": chunk})

# テスト1：特定IDでの検索（BM25が得意なケース）
# report.md に INC-2023-Q4-011 を含むチャンクがないため、
# 代わりに固有の数値「342」で比較する
query_id = "342件のバグ修正"
results_bm25 = bm25_store.search(query_id, top_k=2)

print(f"=== BM25検索: '{query_id}' ===")
for doc, distance in results_bm25:
    print(f"\n距離: {distance:.4f}")
    print(doc["content"][:150])

# テスト2：意味検索との比較（セマンティックが得意なケース）
query_semantic = "プログラマーチームの成果"  # 「342」という単語はないが意味は近い
results_bm25_2 = bm25_store.search(query_semantic, top_k=2)

print(f"\n\n=== BM25検索: '{query_semantic}' ===")
for doc, distance in results_bm25_2:
    print(f"\n距離: {distance:.4f}")
    print(doc["content"][:150])

=== BM25検索: '342件のバグ修正' ===

距離: 0.0000
第2章：ソフトウェアエンジニアリング
当部門は分散システムにおける安定性向上に多大な努力を注いだ。
エンジニアチームは合計342件のバグを修正し、システム稼働率は99.9%を達成した。
新機能として、リアルタイム監視ダッシュボードとAI駆動の異常検知システムを導入した。
コードレビュープロセスの強化

距離: 1.0000
# 年次報告書 2024



=== BM25検索: 'プログラマーチームの成果' ===

距離: 0.0000
第2章：ソフトウェアエンジニアリング
当部門は分散システムにおける安定性向上に多大な努力を注いだ。
エンジニアチームは合計342件のバグを修正し、システム稼働率は99.9%を達成した。
新機能として、リアルタイム監視ダッシュボードとAI駆動の異常検知システムを導入した。
コードレビュープロセスの強化

距離: 0.4953
第1章：医学研究
今年は、XDR-47ウイルスに関する理解において大きな進歩が見られた。
新しいワクチン候補が3件特定され、うち1件は臨床試験フェーズ2に進んだ。
研究チームは感染経路の解明に注力し、従来比40%の感染リスク低減を達成した。



## 7. マルチインデックスRAGパイプライン（ハイブリッド検索）

`VectorIndex`（意味検索）と `BM25Index`（語彙検索）の結果を **相互ランク融合（RRF）** で統合します。

**RRFの計算式：**
```
RRF_score(d) = Σ 1 / (k + rank_i(d))
```
- 各インデックスでのランク順位をスコアに変換して合算
- 複数のインデックスで上位に来た文書ほど高スコア
- `k`（通常60）はランクの影響を滑らかにする定数

In [118]:
class Retriever:
    """複数の検索インデックスをRRFで統合するハイブリッド検索クラス"""

    def __init__(self, *indexes):
        if len(indexes) == 0:
            raise ValueError("インデックスを1つ以上指定してください")
        self._indexes = list(indexes)

    def add_document(self, document: dict):
        """全インデックスに同じ文書を追加する"""
        for index in self._indexes:
            index.add_document(document)

    def search(self, query: str, top_k: int = 2, k_rrf: int = 60) -> list[tuple]:
        """全インデックスで検索してRRFでスコアを統合する"""
        # 全インデックスの検索結果を取得（全件）
        all_results = []
        for index in self._indexes:
            results = index.search(query, top_k=len(index.documents) if hasattr(index, 'documents') else top_k * 3)
            all_results.append(results)

        # RRFスコアを計算（文書テキストをキーに集計）
        rrf_scores = {}
        doc_map = {}

        for results in all_results:
            for rank, (doc, _) in enumerate(results, start=1):
                key = doc["content"][:50]  # 先頭50文字をキーとして使用
                doc_map[key] = doc
                rrf_scores[key] = rrf_scores.get(key, 0) + 1.0 / (k_rrf + rank)

        # スコア降順でソートして上位k件を返す
        sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [(doc_map[key], score) for key, score in sorted_results[:top_k]]


print("Retriever クラスの定義完了")

Retriever クラスの定義完了


In [119]:
# ハイブリッド検索のセットアップ
# VectorIndex と BM25Index を両方持つ Retriever を作成
hybrid_retriever = Retriever(store, bm25_store)

# テスト1：固有数値での検索（BM25が得意）
query1 = "342件のバグ修正"
print(f"=== ハイブリッド検索: '{query1}' ===")
for doc, score in hybrid_retriever.search(query1, top_k=2):
    print(f"\nRRFスコア: {score:.4f}")
    print(doc["content"][:150])

# テスト2：意味での検索（セマンティックが得意）
query2 = "プログラマーチームの成果"
print(f"\n\n=== ハイブリッド検索: '{query2}' ===")
for doc, score in hybrid_retriever.search(query2, top_k=2):
    print(f"\nRRFスコア: {score:.4f}")
    print(doc["content"][:150])

=== ハイブリッド検索: '342件のバグ修正' ===

RRFスコア: 0.0328
第2章：ソフトウェアエンジニアリング
当部門は分散システムにおける安定性向上に多大な努力を注いだ。
エンジニアチームは合計342件のバグを修正し、システム稼働率は99.9%を達成した。
新機能として、リアルタイム監視ダッシュボードとAI駆動の異常検知システムを導入した。
コードレビュープロセスの強化

RRFスコア: 0.0323
# 年次報告書 2024



=== ハイブリッド検索: 'プログラマーチームの成果' ===

RRFスコア: 0.0328
第2章：ソフトウェアエンジニアリング
当部門は分散システムにおける安定性向上に多大な努力を注いだ。
エンジニアチームは合計342件のバグを修正し、システム稼働率は99.9%を達成した。
新機能として、リアルタイム監視ダッシュボードとAI駆動の異常検知システムを導入した。
コードレビュープロセスの強化

RRFスコア: 0.0318
概要
本報告書は、各部門の2024年度の活動実績をまとめたものです。

